# HireMind: LoRA fine-tuning on Colab

Fine-tunes Qwen3-0.6B on the Parser, Critic and Searcher examples (`training/train_lora.py`).

1. **Runtime > Change runtime type > GPU** (a T4 is enough).
2. Run the cells in order. Checkpoints are saved to the shared Drive folder, so if Colab disconnects, run the **Resume** cell.
3. The best checkpoint (lowest validation loss) is saved to `models/slm/qwen3-0.6b-lora/final`.

Hassaan: the shared folder is under *Shared with me*. Right-click `HireMind-shared` > **Organize > Add shortcut** to My Drive first, so the path below works.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SHARED = '/content/drive/MyDrive/HireMind-shared'
DATA = f'{SHARED}/data/processed/training'
OUT = f'{SHARED}/models/slm/qwen3-0.6b-lora'
!ls -la "{DATA}"

Get the code. If the repository is private, use a GitHub token in the URL or upload the repo instead.

In [ ]:
!git clone -q https://github.com/arsalqazi3/HireMind.git /content/HireMind
%cd /content/HireMind
!git log --oneline -1

Install the library versions the training script was tested with. Colab's own PyTorch is kept.

In [ ]:
!pip install -q "transformers==5.20.1" "peft==0.21.2" "accelerate==1.15.0"

## Train

Uses all Parser and Searcher examples and 6,000 sampled Critic pairs for 2 epochs. The first line printed gives the real number of tokens per epoch. Use it to judge the time, and lower `--critic-limit` or `--epochs` if it is too much for one session.

In [ ]:
!python -m training.train_lora --data "{DATA}" --out "{OUT}"

## Resume after a disconnect

In [ ]:
!python -m training.train_lora --data "{DATA}" --out "{OUT}" --resume

## Quick check

Generates one validation Parser answer with the fine-tuned adapter and prints the expected answer next to it. This is a sanity check, not an evaluation.

In [ ]:
import json, torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
from hiremind import config

final = f'{OUT}/final'
tokenizer = AutoTokenizer.from_pretrained(final)
model = AutoPeftModelForCausalLM.from_pretrained(final).to('cuda').eval()
example = json.loads(open(f'{DATA}/parser_val.jsonl', encoding='utf-8').readline())
text = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': f"{config.TASK_TOKENS[example['task']]}
{example['prompt']}"}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(text, return_tensors='pt').to('cuda')
with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=256, do_sample=False)
print('model:   ', tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True))
print('expected:', example['completion'])